# Lesson 01 - 人工智能代理簡介

歡迎來到 **初學者人工智能代理** 課程的第一課！

**人工智能代理** 是一個以大型語言模型（LLM）為推理引擎的程式，能夠在現實世界中採取*行動*——呼叫 API、查詢資料庫或執行程式碼——以代表用戶達成目標。

在這份筆記本中，你將建立你的第一個代理：一個推薦度假目的地的 **旅遊代理**。在此過程中，你將學習如何：

1. 使用 **Microsoft Agent Framework** 連接到 Azure AI Foundry Agent 服務。
2. 給代理一個 **工具** —— 一個它可以呼叫的純 Python 函數。
3. 執行代理並檢查它的回應。
4. 逐個標記（token）串流代理的回應。


## Setup

在運行此筆記本之前，請確保您已經：

1. **擁有一個已部署聊天模型的 Azure AI Foundry 項目**（例如 `gpt-4o-mini`）。
2. **已使用 Azure CLI 登錄** — 在您的終端機中運行 `az login`。
3. **設置所需的環境變量：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 項目端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名稱。

下面的代碼單元會安裝您需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 創建你的第一個代理人

代理人需要兩樣東西：

- **指令**，告訴它 *它是誰* 以及 *如何行為*（系統提示）。
- **工具** — 使用 `@tool` 裝飾的 Python 函數，代理人可以調用這些工具來檢索資訊或執行操作。

下面我們定義了一個簡單的工具，返回熱門度假目的地的清單。當用戶詢問旅遊建議時，代理人將使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式回應

為了更互動的體驗，你可以**串流**代理的回應。代理會隨著文字產生即時地輸出文字片段，而不用等待整段回應完成。這在聊天介面中特別有用，可以即時顯示輸出結果。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課中你學會了如何：

- **建立一個提供者**，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent Service。
- **使用 `@tool` 裝飾器定義工具**，讓代理可以呼叫你的 Python 函數。
- **運行代理**，傳入使用者訊息並打印其回應。
- **串流回應**，實現即時輸出。

在下一課中，我們將更深入探討智能代理框架，並學習如何賦予代理更強大的工具與多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件是使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於重要資訊，建議使用專業人工翻譯。本公司不對因使用本翻譯所引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
